In [ ]:
!pip install langchain-classic
!pip install langchain_google_genai
!pip install google-generativeai

In [ ]:
import os
from google import genai
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate

GOOGLE_API_KEY = 'YOUR_API_KEY'
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

client = genai.Client()

print("Models available to your API key:\n" + "-"*35)
for model in client.models.list():
    if model.supported_actions and "generateContent" in model.supported_actions:
        clean_name = model.name.replace("models/", "")
        print(f"• {clean_name}")

llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash", temperature=0)

def create_chain(prompt_template_str):
    prompt = PromptTemplate.from_template(prompt_template_str)
    return prompt | llm

def get_response(prompt):
  return llm.invoke(prompt).content

### **Basic Prompt Structures**

**Single-turn Prompts**

In [ ]:
single_turn_prompt = "What are the three primary colors?"
print(llm.invoke(single_turn_prompt).content)

In [ ]:
structured_prompt = PromptTemplate(
    input_variables=["topic"],
    template="Provide a brief explanation of {topic} and list its three main components."
)

chain = structured_prompt | llm
result = chain.invoke({"topic": "color theory"}).content
print(result)

**Multi-turn Prompts (Conversations)**

In [ ]:
from langchain_classic.chains import ConversationChain
from langchain_classic.memory import ConversationBufferMemory

conversation = ConversationChain(
    llm=llm,
    verbose=True,
    memory=ConversationBufferMemory()
)

print(conversation.invoke(input="Hi, I'm learning about space. Can you tell me about planets?")["response"])
print(conversation.invoke(input="What's the largest planet in our solar system?")["response"])
print(conversation.invoke(input="How does its size compare to Earth?")["response"])

In [ ]:
prompts = [
    "What is the capital of France?",
    "What is its population?",
    "What is the city's most famous landmark?"
]

print("Single-turn responses:")
for prompt in prompts:
    print(f"Q: {prompt}")
    print(f"A: {llm.invoke(prompt).content}\n")

print("Multi-turn responses:")
conversation = ConversationChain(llm=llm, memory=ConversationBufferMemory())
for prompt in prompts:
    print(f"Q: {prompt}")
    print(f"A: {conversation.invoke(input=prompt)['response']}\n")

## **Zero-Shot Prompting**

In [ ]:
direct_task_prompt = """Classify the sentiment of the following text as positive, negative, or neutral.
Do not explain your reasoning, just provide the classification.

Text: {text}

Sentiment:"""

direct_task_chain = create_chain(direct_task_prompt)

texts = [
    "I absolutely loved the movie! The acting was superb.",
    "The weather today is quite typical for this time of year.",
    "I'm disappointed with the service I received at the restaurant."
]

for text in texts:
    result = direct_task_chain.invoke({"text": text}).content
    if isinstance(result, list):
        result_text = "".join([part if isinstance(part, str) else str(part) for part in result])
    else:
        result_text = str(result)

    print(f"Text: {text}\nSentiment: {result_text.strip()}\n")

In [ ]:
format_spec_prompt = """Generate a short news article about {topic}.
Structure your response in the following format:

Headline: [A catchy headline for the article]

Lead: [A brief introductory paragraph summarizing the key points]

Body: [2-3 short paragraphs providing more details]

Conclusion: [A concluding sentence or call to action]"""

format_spec_chain = create_chain(format_spec_prompt)
topic = "The discovery of a new earth-like exoplanet"
result = format_spec_chain.invoke({"topic": topic}).content
print(result)

In [ ]:
multi_step_prompt = """Analyze the following text for its main argument, supporting evidence, and potential counterarguments.
Provide your analysis in the following steps:

1. Main Argument: Identify and state the primary claim or thesis.
2. Supporting Evidence: List the key points or evidence used to support the main argument.
3. Potential Counterarguments: Suggest possible objections or alternative viewpoints to the main argument.

Text: {text}

Analysis:"""

multi_step_chain = create_chain(multi_step_prompt)

text =  """While electric vehicles are often touted as a solution to climate change, their environmental impact is not as straightforward as it seems.
The production of batteries for electric cars requires significant mining operations, which can lead to habitat destruction and water pollution.
Moreover, if the electricity used to charge these vehicles comes from fossil fuel sources, the overall carbon footprint may not be significantly reduced.
However, as renewable energy sources become more prevalent and battery technology improves, electric vehicles could indeed play a crucial role in combating climate change."""

result = multi_step_chain.invoke({"text": text}).content
print(result)

In [ ]:
def compare_prompts(task, prompt_templates):
  print(f"Task: {task}\n")
  for name, template in prompt_templates.items():
    chain = create_chain(template)
    result = chain.invoke({"task": task}).content
    print(f"{name} Prompt Result:")
    print(result)
    print("\n" + "-"*50 + "\n")

task = "Explain concisely the concept of blockchain technology"

prompt_templates = {
    "Basic": "Explain {task}.",
    "Structured": """Explain {task} by addressing the following points:
1. Definition
2. Key features
3. Real-world applications
4. Potential impact on industries"""
}

compare_prompts(task, prompt_templates)

## **Few-Shot Learning**

In [ ]:
def few_shot_sentiment_classification(input_text):
  few_shot_prompt = PromptTemplate(
      input_variables=["input_text"],
      template="""
        Classify the sentiment as Positive, Negative, or Neutral.

        Examples:
        Text: I love this product! It's amazing.
        Sentiment: Positive

        Text: This movie was terrible. I hated it.
        Sentiment: Negative

        Text: The weather today is okay.
        Sentiment: Neutral

        Now, classify the following:
        Text: {input_text}
        Sentiment:
        """
  )

  chain = few_shot_prompt | llm
  raw_result = chain.invoke(input_text).content
  if isinstance(raw_result, list):
    result = "".join([part["text"] if isinstance(part, dict) and "text" in part else str(part) for part in raw_result])
  else:
    result = str(raw_result)

  result = result.strip()
  if ':' in result:
    result = result.split(':')[-1].strip()
  return result

test_text = "I can't believe how great this new restaurant is!"
result = few_shot_sentiment_classification(test_text)
print(f"Input: {test_text}")
print(f"Predicted Sentiment: {result}")

In [ ]:
def multi_task_few_shot(input_text, task):
  few_shot_prompt = PromptTemplate(
      input_variables=["input_text", "task"],
      template="""
        Perform the specified task on the given text.

        Examples:
        Text: I love this product! It's amazing.
        Task: sentiment
        Result: Positive

        Text: Bonjour, comment allez-vous?
        Task: language
        Result: French

        Now, perform the following task:
        Text: {input_text}
        Task: {task}
        Result:
        """
  )

  chain = few_shot_prompt | llm
  return chain.invoke({"input_text": input_text, "task": task}).content

print(multi_task_few_shot("I can't believe how great this is!", "sentiment"))
print(multi_task_few_shot("Guten Tag, wie geht es Ihnen?", "language"))

In [ ]:
def in_context_learning(task_description, examples, input_text):
  example_text = "".join([f"Input: {e['input']}\nOutput: {e['output']}\n\n" for e in examples])
  in_context_prompt = PromptTemplate(
      input_variables=["task_description", "examples", "input_text"],
      template="""
      Task: {task_description}

      Examples:
      {examples}

      Now, perform the task on the following input:
      Input: {input_text}
      Output:
      """
  )

  chain = in_context_prompt | llm
  return chain.invoke({"task_description": task_description, "examples": example_text, "input_text": input_text}).content

task_desc = "Convert the given text to pig latin."
examples = [
    {"input": "hello", "output": "ellohay"},
    {"input": "apple", "output": "appleay"}
]
test_input = "python"

result = in_context_learning(task_desc, examples, test_input)
print(f"Input: {test_input}")
print(f"Output: {result}")

In [ ]:
def evaluate_model(model_func, test_cases):
  correct = 0
  total = len(test_cases)

  for cases in test_cases:
    input_text = cases["input"]
    expected_output = cases["label"]
    predicted_output = model_func(input_text).strip()

    is_correct = (predicted_output == expected_output)
    correct += int(is_correct)

    print(f"Input: {input_text}")
    print(f"Expected Output: {expected_output}")
    print(f"Predicted Output: {predicted_output}")
    print(f"Correct: {is_correct}\n")

  accuracy = (correct / total) * 100
  print(f"Accuracy: {accuracy:.2f}%")
  return accuracy

test_cases = [
    {"input": "This product exceeded my expectations!", "label": "Positive"},
    {"input": "I'm utterly disappointed with the service.", "label": "Negative"},
    {"input": "The temperature today is 72 degrees.", "label": "Neutral"}
]

accuracy = evaluate_model(few_shot_sentiment_classification, test_cases)
print(f"Model Accuracy: {accuracy:.2f}")

### **Chain of Thought(CoT) Prompting**

In [ ]:
standard_prompt = PromptTemplate(
    input_variables=["question"],
    template="Answer the following question concisely: {question}."
)

cot_prompt = PromptTemplate(
    input_variables=["question"],
    template="Answer the following question step by step concisely: {question}."
)

standard_chain = standard_prompt | llm
cot_chain = cot_prompt | llm

question = "If a train travels 120 km in 2 hours, what is its average speed in km/h?"

standard_response = standard_chain.invoke(question).content
cot_response = cot_chain.invoke(question).content

print("Standard Response:")
print(standard_response)
print("\nChain of Thought Response:")
print(cot_response)

In [ ]:
advanced_cot_prompt = PromptTemplate(
    input_variables=["question"],
    template="""Solve the following problem step by step. For each step:
1. State what you're going to calculate
2. Write the formula you'll use (if applicable)
3. Perform the calculation
4. Explain the result

Question: {question}

Solution:"""
)

advanced_cot_chain = advanced_cot_prompt | llm

complex_question = "A car travels 150 km at 60 km/h, then another 100 km at 50 km/h. What is the average speed for the entire journey?"

advanced_cot_response = advanced_cot_chain.invoke(complex_question).content
print(advanced_cot_response)

In [ ]:
challenging_question = """
A cylindrical water tank with a radius of 1.5 meters and a height of 4 meters is 2/3 full.
If water is being added at a rate of 10 liters per minute, how long will it take for the tank to overflow?
Give your answer in hours and minutes, rounded to the nearest minute.
(Use 3.14159 for π and 1000 liters = 1 cubic meter)"""

standard_response = standard_chain.invoke(challenging_question).content
cot_response = cot_chain.invoke(challenging_question).content

print("Standard Response:")
print(standard_response)
print("\nChain of Thought Response:")
print(cot_response)

In [ ]:
logical_reasoning_prompt = PromptTemplate(
    input_variables=["scenario"],
    template="""Analyze the following logical puzzle thoroughly. Follow these steps in your analysis:

List the Facts:

Summarize all the given information and statements clearly.
Identify all the characters or elements involved.
Identify Possible Roles or Conditions:

Determine all possible roles, behaviors, or states applicable to the characters or elements (e.g., truth-teller, liar, alternator).
Note the Constraints:

Outline any rules, constraints, or relationships specified in the puzzle.
Generate Possible Scenarios:

Systematically consider all possible combinations of roles or conditions for the characters or elements.
Ensure that all permutations are accounted for.
Test Each Scenario:

For each possible scenario:
Assume the roles or conditions you've assigned.
Analyze each statement based on these assumptions.
Check for consistency or contradictions within the scenario.
Eliminate Inconsistent Scenarios:

Discard any scenarios that lead to contradictions or violate the constraints.
Keep track of the reasoning for eliminating each scenario.
Conclude the Solution:

Identify the scenario(s) that remain consistent after testing.
Summarize the findings.
Provide a Clear Answer:

State definitively the role or condition of each character or element.
Explain why this is the only possible solution based on your analysis.
Scenario:

{scenario}

Analysis:"""
)

logical_resoning_chain = logical_reasoning_prompt | llm
logical_puzzle = """In a room, there are three people: Amy, Bob, and Charlie.
One of them always tells the truth, one always lies, and one alternates between truth and lies.
Amy says, 'Bob is a liar.'
Bob says, 'Charlie alternates between truth and lies.'
Charlie says, 'Amy and I are both liars.'
Determine the nature (truth-teller, liar, or alternator) of each person."""

logical_resoning_response = logical_resoning_chain.invoke(logical_puzzle).content
print(logical_resoning_response)

### **Self-Consistency**

In [ ]:
def generate_multiple_paths(problem, num_paths=3):
  prompt_template = PromptTemplate(
      input_variables=["problem", "path_number"],
      template="""Solve the following problem using a unique approach. This is reasoning path {path_number}.
        Problem: {problem}
        Reasoning path {path_number}:"""
  )

  paths = []
  for i in range(1, num_paths):
    chain = prompt_template | llm
    path = chain.invoke({"problem": problem, "path_number": i+1}).content
    paths.append(path)

  return paths

In [ ]:
problem = "A ball is thrown upwards with an initial velocity of 20 m/s. How high will it go?"
paths = generate_multiple_paths(problem)

for i, path in enumerate(paths, 1):
  print(f"Path {i}:\n{path}\n")

In [ ]:
def aggregate_results(paths):
  prompt_template = PromptTemplate(
      input_variables=["paths"],
      template="""Analyze the following reasoning paths and determine the most consistent answer. If there are discrepancies, explain why and provide the most likely correct answer.
        Reasoning paths:
        {paths}

        Most consistent answer:"""
  )

  chain = prompt_template | llm
  return chain.invoke({"paths": "\n".join(paths)}).content

aggregated_result = aggregate_results(paths)
print("Aggregated Result:\n", aggregated_result)

In [ ]:
def self_consistency_check(problem, aggregated_result):
  prompt_template = PromptTemplate(
      input_variables=["problem", "result"],
      template="""Evaluate the consistency and reliability of the following result for the given problem.
        Problem: {problem}
        Result: {result}

        Evaluation (consider factors like logical consistency, adherence to known facts, and potential biases):"""
  )

  chain = prompt_template | llm
  return chain.invoke({"problem": problem, "result": aggregated_result}).content

consistency_evaluation = self_consistency_check(problem, aggregated_result)
print("Self-Consistency Evaluation:\n", consistency_evaluation)

In [ ]:
def solve_problem(problem):
  paths = generate_multiple_paths(problem)
  aggregated_result = aggregate_results(paths)
  consistency_evaluation = self_consistency_check(problem, aggregated_result)
  return aggregated_result, consistency_evaluation

problems = [
    "What is the capital of France?",
    "Explain the concept of supply and demand in economics.",
    "If a train travels at 60 km/h, how long will it take to cover 180 km?"
]

for problem in problems:
    print(f"Problem: {problem}")
    result, evaluation = solve_problem(problem)
    print("Aggregated Result:\n", result)
    print("\nConsistency Evaluation:\n", evaluation)
    print("\n" + "-"*50 + "\n")

### **Negative Prompting**

In [ ]:
negative_prompt = PromptTemplate(
    input_variables=["topic"],
    template="""Provide a brief explanation of {topic}.
    Do NOT include any of the following in your explanation:
    - Technical jargon or complex terminology
    - Historical background or dates
    - Comparisons to other related topics
    Your explanation should be simple, direct, and focus only on the core concept."""
)

response = get_response(negative_prompt.format(topic="photosynthesis"))
print(response)

In [ ]:
exclusion_prompt = PromptTemplate(
    input_variables=["topic", "exclude"],
    template="""Write a short paragraph about {topic}.
    Important: Do not mention or reference anything related to {exclude}."""
)

response = get_response(exclusion_prompt.format(topic="the benefits of exercise", exclude="weight loss or body image"))
print(response)

In [ ]:
constraint_prompt = PromptTemplate(
    input_variables=["topic", "style", "excluded_words"],
    template="""Write a {style} description of {topic}.
    Constraints:
    1. Do not use any of these words: {excluded_words}
    2. Keep the description under 100 words
    3. Do not use analogies or metaphors
    4. Focus only on factual information"""
)

response = get_response(constraint_prompt.format(
    topic="artificial intelligence",
    style="technical",
    excluded_words="robot, human-like, science fiction"))
print(response)

In [ ]:
import re

def evaluate_output(output, constraints):
    """Evaluate if the output meets the given constraints."""
    results = {}
    for constraint, check_func in constraints.items():
        results[constraint] = check_func(output)
    return results

constraints = {
    "word_count": lambda x: len(x.split()) <= 100,
    "no_excluded_words": lambda x: all(word not in x.lower() for word in ["robot", "human-like", "science fiction"]),
    "no_analogies": lambda x: not re.search(r"\b(as|like)\b", x, re.IGNORECASE)
}

In [ ]:
evaluation_results = evaluate_output(response, constraints)
print("Evaluation results:", evaluation_results)

if not all(evaluation_results.values()):
    refined_prompt = constraint_prompt.format(
        topic="artificial intelligence",
        style="technical and concise",
        excluded_words="robot, human-like, science fiction, like, as"
    )
    refined_response = get_response(refined_prompt)
    print("\nRefined response:\n", refined_response)

    refined_evaluation = evaluate_output(refined_response, constraints)
    print("\nRefined evaluation results:", refined_evaluation)

### **Multilingual and Cross-lingual Prompting**

In [ ]:
multilingual_greeting = PromptTemplate(
    input_variables=["language"],
    template="Greet the user in {language} and provide a short introduction about the weather in a country where this language is spoken."
)

languages = ["English", "Bangla", "French", "German", "Japanese"]

for lang in languages: # Fixed typo 'laguages'
  prompt = multilingual_greeting.format(language=lang)
  response = get_response(prompt)
  print(f"Response in {lang}:\n{response}\n")

In [ ]:
language_adaptive_prompt = PromptTemplate(
    input_variables=["user_input"],
    template="""Detect the language of the following input and respond in the same language:
    User input: {user_input}
    Your response (in the detected language):"""
)

inputs = [
    "Hello, how are you?",
    "হে , কেমন আছো?",
    "Bonjour, comment allez-vous ?",
    "こんにちは、お元気ですか？",
    "Здравствуйте, как дела?"
]

for user_input in inputs:
    prompt = language_adaptive_prompt.format(user_input=user_input)
    response = llm.invoke(prompt)
    print(f"Input: {user_input}")
    print("Response:")
    print(response.content)
    print()

In [ ]:
translation_prompt = PromptTemplate(
    input_variables=["source_lang", "target_lang", "text"],
    template="""Translate the following text from {source_lang} to {target_lang}:
    {source_lang} text: {text}
    {target_lang} translation:"""
)

translations = [
    {"source_lang": "English", "target_lang": "French", "text": "The quick brown fox jumps over the lazy dog."},
    {"source_lang": "Spanish", "target_lang": "Bangla", "text": "La vida es bella."},
    {"source_lang": "Japanese", "target_lang": "English", "text": "桜の花が満開です。"}
]

for t in translations:
    prompt = translation_prompt.format(**t)
    response = llm.invoke(prompt)
    print(f"From {t['source_lang']} to {t['target_lang']}:")
    print(f"Original: {t['text']}")
    print("Translation:")
    print(response.content)
    print()

In [ ]:
non_latin_prompt = PromptTemplate(
    input_variables=["text", "script"],
    template="""Provide the following information for the given text:
    1. The original text
    2. The name of the script/writing system
    3. A transliteration to Latin alphabet
    4. An English translation

    Text: {text}
    Script: {script}
    """
)

non_latin_texts = [
    {"text": "こんにちは、世界", "script": "Japanese"},
    {"text": "Здравствуй, мир", "script": "Cyrillic"},
    {"text": "কেমন আছো?", "script": "Bengali"}
]

for text in non_latin_texts:
    prompt = non_latin_prompt.format(**text)
    response = llm.invoke(prompt)
    print(response.content)
    print()

In [ ]:
cultural_translation_prompt = PromptTemplate(
    input_variables=["source_lang", "target_lang", "text"],
    template="""Translate the following text from {source_lang} to {target_lang}, paying special attention to cultural context and idiomatic expressions. Provide:
    1. A direct translation
    2. A culturally adapted translation (if different)
    3. Explanations of any cultural nuances or idioms

    {source_lang} text: {text}
    {target_lang} translation and explanation:"""
)

cultural_texts = [
    {"source_lang": "English", "target_lang": "Japanese", "text": "It's raining cats and dogs."},
    {"source_lang": "French", "target_lang": "English", "text": "Je suis dans le pétrin."},
    {"source_lang": "Spanish", "target_lang": "German", "text": "Cuesta un ojo de la cara."}
]

for text in cultural_texts:
    prompt = cultural_translation_prompt.format(**text)
    response = llm.invoke(prompt)
    print(f"From {text['source_lang']} to {text['target_lang']}:")
    print(f"Original: {text['text']}")
    print("Translation and Explanation:")
    print(response.content)
    print()